# 02 - Data Cleaning

**Objective:** Address the issues identified during the Data Understanding stage to create a clean dataset ready for modeling.

> **Principle:** Only handle the issues that have been identified. As confirmed in NB01: there are no missing NaN values and no duplicate records.

**Steps:**
1. Load and sort the data chronologically
2. Remove unnecessary columns
3. Scale numerical variables by Match Week (MW)
4. Apply One-Hot Encoding for HM1–HM3 and AM1–AM3
5. Encode the target variable FTR
6. Perform final checks and save the dataset

## 1. Import & Load data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw_data.csv', index_col=0)
print(f'Shape ban đầu: {df.shape}')
df.head(3)

Shape ban đầu: (6840, 39)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTGS,ATGS,HTGC,ATGC,...,HTLossStreak3,HTLossStreak5,ATWinStreak3,ATWinStreak5,ATLossStreak3,ATLossStreak5,HTGD,ATGD,DiffPts,DiffFormPts
0,19/08/00,Charlton,Man City,4,0,H,0,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,19/08/00,Chelsea,West Ham,4,2,H,0,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
2,19/08/00,Coventry,Middlesbrough,1,3,NH,0,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0


## 2. Sort data by time

> Sorting data in chronological order is mandatory before splitting into train/test sets. If the data is not properly sorted, splitting by index will not preserve temporal order, which can lead to data leakage.

In [3]:
# Chuyển Date sang datetime và sắp xếp
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

print(f'Trận đầu tiên: {df["Date"].iloc[0].date()}')
print(f'Trận cuối cùng: {df["Date"].iloc[-1].date()}')
print(f'Tổng số trận: {len(df)}')

Trận đầu tiên: 2000-08-19
Trận cuối cùng: 2018-05-13
Tổng số trận: 6840


## 3. Remove unnecessary columns

In [4]:
cols_to_drop = [
    # Định danh — không có giá trị dự đoán
    'HomeTeam', 'AwayTeam',
    # Chuỗi dạng 'WWDLL' — đã tổng hợp thành số ở HTFormPts, ATFormPts
    'HTFormPtsStr', 'ATFormPtsStr',
    # DATA LEAKAGE: FTHG, FTAG là kết quả trận — biết trước khi dự đoán là gian lận
    'FTHG', 'FTAG',
    # DƯ THỪA: HTGS, ATGS, HTGC, ATGC đã được tổng hợp vào HTGD (= HTGS - HTGC)
    # và gián tiếp vào HTP, ATP — giữ lại gây multicollinearity
    'HTGS', 'ATGS', 'HTGC', 'ATGC',
    # ÍT THÔNG TIN: kết quả cách đây 4-5 trận ít tương quan hơn 3 trận gần nhất
    'HM4', 'HM5', 'AM4', 'AM5'
]

df = df.drop(columns=cols_to_drop)
print(f'Đã bỏ {len(cols_to_drop)} cột.')
print(f'Shape sau khi bỏ cột: {df.shape}')
print(f'\nCác cột còn lại: {list(df.columns)}')

Đã bỏ 14 cột.
Shape sau khi bỏ cột: (6840, 25)

Các cột còn lại: ['Date', 'FTR', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'AM1', 'AM2', 'AM3', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']


**Detailed explanation of each group:**

| Group | Columns | Reason for removal |
|:---|:---|:---|
| Identification | `HomeTeam`, `AwayTeam` | Do not provide direct predictive information |
| Redundant | `HTFormPtsStr`, `ATFormPtsStr` | Numeric versions `HTFormPts`, `ATFormPts` already exist |
| **Data leakage** | `FTHG`, `FTAG` | Match results not available at prediction time |
| Redundant | `HTGS`, `ATGS`, `HTGC`, `ATGC` | Already aggregated into `HTGD` = HTGS − HTGC, and indirectly reflected in `HTP`/`ATP` |
| Low information | `HM4`, `HM5`, `AM4`, `AM5` | Results from 4–5 previous matches have weaker correlation than the last 3 matches |

> **Keep `Date`** at this stage for time-based train/test splitting in NB03. It will be dropped before feeding into the model.

## 4. Scale numerical variables by Match Week (MW)

> **Why is scaling by MW necessary?**
>
> Cumulative variables such as `HTP`, `HTGD`, `DiffPts`, etc. naturally increase over the course of a season:
> - **30 points at Week 10** → ~3 points per match (very strong performance)
> - **30 points at Week 30** → ~1 point per match (only average performance)
>
> → Therefore, dividing by MW converts these features into per-match average indicators, ensuring fair comparability across different stages of the season.

In [5]:
scale_cols = ['HTGD', 'ATGD', 'DiffPts', 'DiffFormPts', 'HTP', 'ATP']
df['MW'] = df['MW'].astype(float)

print('Ví dụ trước khi scale (tuần 10):')
print(df[df['MW'] == 10][scale_cols + ['MW']].head(2).to_string())

for col in scale_cols:
    df[col] = df[col] / df['MW']

print('\nSau khi scale (tuần 10) — giá trị trung bình/trận:')
print(df[df['MW'] == 10][scale_cols].head(2).to_string())

# Bỏ MW sau khi đã dùng xong
df = df.drop(columns=['MW'])
print('\n Scale hoàn tất. Đã bỏ cột MW.')

Ví dụ trước khi scale (tuần 10):
    HTGD  ATGD  DiffPts  DiffFormPts  HTP  ATP    MW
89   0.3   0.2     -0.1          0.0  1.5  1.6  10.0
90  -0.1   0.0      0.2          0.3  1.2  1.0  10.0

Sau khi scale (tuần 10) — giá trị trung bình/trận:
    HTGD  ATGD  DiffPts  DiffFormPts   HTP   ATP
89  0.03  0.02    -0.01         0.00  0.15  0.16
90 -0.01  0.00     0.02         0.03  0.12  0.10

 Scale hoàn tất. Đã bỏ cột MW.


## 5. One-Hot Encoding for HM1–HM3, AM1–AM3

> **Why use One-Hot Encoding instead of W=3 / D=1 / L=0?**
>
> W/D/L are nominal variables — they do not have a true mathematical linear order. If encoded as W=3, D=1, L=0, the model assumes that the distance from W→D is equal to D→L = 1, which is semantically incorrect.
>
> One-Hot Encoding creates independent binary columns, allowing the model to learn separate weights for each outcome.
> `'M'` (insufficient match history) is kept as a separate category instead of being forced into 0 (loss).

In [6]:
result_cols_ohe = ['HM1', 'HM2', 'HM3', 'AM1', 'AM2', 'AM3']

print('Giá trị duy nhất trong HM1:', sorted(df['HM1'].unique()))

for col in result_cols_ohe:
    df[col] = df[col].astype(str)

df = pd.get_dummies(df, columns=result_cols_ohe, prefix=result_cols_ohe)

ohe_cols = [c for c in df.columns if any(c.startswith(p+'_') for p in result_cols_ohe)]
print(f'\nCác cột OHE mới ({len(ohe_cols)} cột):')
print(ohe_cols)
print(f'\nShape sau OHE: {df.shape}')

Giá trị duy nhất trong HM1: ['D', 'L', 'M', 'W']

Các cột OHE mới (24 cột):
['HM1_D', 'HM1_L', 'HM1_M', 'HM1_W', 'HM2_D', 'HM2_L', 'HM2_M', 'HM2_W', 'HM3_D', 'HM3_L', 'HM3_M', 'HM3_W', 'AM1_D', 'AM1_L', 'AM1_M', 'AM1_W', 'AM2_D', 'AM2_L', 'AM2_M', 'AM2_W', 'AM3_D', 'AM3_L', 'AM3_M', 'AM3_W']

Shape sau OHE: (6840, 42)


**Observation:** 6 columns × 4 values (W/D/L/M) = 24 new binary columns. The model can learn the separate impact of each recent match outcome.

## 6. Encode the target variable FTR

In [7]:
df['FTR'] = df['FTR'].map({'H': 1, 'NH': 0})
print('Phân phối FTR sau encode:')
print(df['FTR'].value_counts())
print(f'\nTỷ lệ Home Win: {df["FTR"].mean()*100:.1f}%')

Phân phối FTR sau encode:
FTR
0    3664
1    3176
Name: count, dtype: int64

Tỷ lệ Home Win: 46.4%


## 🔹 7. Final result check

In [8]:
print('=== THÔNG TIN SAU KHI LÀM SẠCH ===')
print(f'Shape:          {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('Tất cả cột:')
print(list(df.columns))
print()
df.head(3)

=== THÔNG TIN SAU KHI LÀM SẠCH ===
Shape:          (6840, 42)
Missing values: 0
Duplicate rows: 113

Tất cả cột:
['Date', 'FTR', 'HTP', 'ATP', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts', 'HM1_D', 'HM1_L', 'HM1_M', 'HM1_W', 'HM2_D', 'HM2_L', 'HM2_M', 'HM2_W', 'HM3_D', 'HM3_L', 'HM3_M', 'HM3_W', 'AM1_D', 'AM1_L', 'AM1_M', 'AM1_W', 'AM2_D', 'AM2_L', 'AM2_M', 'AM2_W', 'AM3_D', 'AM3_L', 'AM3_M', 'AM3_W']



,Date,FTR,HTP,ATP,HTFormPts,ATFormPts,HTWinStreak3,HTWinStreak5,HTLossStreak3,HTLossStreak5,...,AM1_M,AM1_W,AM2_D,AM2_L,AM2_M,AM2_W,AM3_D,AM3_L,AM3_M,AM3_W
0,2000-08-19,1,0.0,0.0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False
1,2000-08-19,1,0.0,0.0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False
2,2000-08-19,0,0.0,0.0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False


## 8. Save the cleaned dataset

In [10]:
df.to_csv('../data/cleaned_data.csv', index=False)
print('Đã lưu: ../data/cleaned_data.csv')
print(f'   Shape: {df.shape}')
print(f'   Gồm: Date (để chia train/test), FTR (target Classification), và {len(df.columns)-2} features')

Đã lưu: ../data/cleaned_data.csv
   Shape: (6840, 42)
   Gồm: Date (để chia train/test), FTR (target Classification), và 40 features


## Summary

| Step | Action | Result |
|:---|:---|:---|
| Sort by time | `pd.to_datetime` + `sort_values('Date')` | Ensures correct order for time-based split |
| Drop columns | 14 columns (leakage + redundant + low-information) | 39 → 25 columns (before OHE) |
| Scale by MW | Divide 6 cumulative variables by MW | Normalized across season stages |
| One-Hot Encoding | HM1–HM3, AM1–AM3 → 24 binary columns | Proper encoding for nominal variables |
| Encode FTR | H=1, NH=0 | Classification target variable |

**Conclusion:** Clean dataset contains **6,840 rows × 43 columns** (including `Date` and `FTR`), ready for modeling.